# Chapter 8: 量化建模 (Quantitative Modeling)

构建价格预测模型（Random Forest + XGBoost）和 K-Means 市场细分模型，
验证车辆参数对价格的预测能力，通过特征重要性分析量化各参数的贡献度。

## 8.0 环境设置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import sys
import pickle

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

# 设置字体
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 导入模型库
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from xgboost import XGBRegressor

# 路径配置
DATA_FILE = os.path.join(project_root, 'outputs', 'ch01_data_cleaning', 'cleaned_data.csv')
OUTPUT_DIR = os.path.join(project_root, 'outputs', 'ch08_quantitative_modeling')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"数据文件: {DATA_FILE}")
print(f"输出目录: {OUTPUT_DIR}")

## 8.1 特征工程

In [ ]:
# 加载数据
df = pd.read_csv(DATA_FILE)
print(f"原始数据形状: {df.shape}")

# 定义特征列
cat_cols = ['brand', 'drive_type', 'body_type', 'country_of_origin', 'market_segment']
num_cols = ['battery_capacity_kwh', 'range_miles', 'charging_speed_kw',
            'acceleration_0_60_mph', 'top_speed_mph', 'horsepower', 'torque_nm',
            'weight_kg', 'seating_capacity', 'cargo_volume_cubic_ft',
            'safety_rating', 'autopilot_level', 'warranty_years',
            'price_per_kwh', 'efficiency', 'power_to_weight']
target_col = 'price_usd'

print(f"\n数值特征: {len(num_cols)}")
print(f"分类特征: {len(cat_cols)}")

# 检查缺失值
print(f"\n目标变量缺失值: {df[target_col].isna().sum()}")
print(f"数值特征缺失值总计: {df[num_cols].isna().sum().sum()}")
print(f"分类特征缺失值总计: {df[cat_cols].isna().sum().sum()}")

In [ ]:
# One-Hot 编码分类变量
df_cat_encoded = pd.get_dummies(df[cat_cols], drop_first=False, dtype=int)
print(f"One-Hot 编码后分类特征数: {df_cat_encoded.shape[1]}")

# StandardScaler 标准化数值变量
scaler = StandardScaler()
df_num_scaled = pd.DataFrame(
    scaler.fit_transform(df[num_cols]),
    columns=num_cols,
    index=df.index
)

# 合并特征矩阵
X = pd.concat([df_num_scaled, df_cat_encoded], axis=1)
y = df[target_col]

feature_names = list(X.columns)
print(f"\n特征矩阵形状: {X.shape}")
print(f"特征数量: {len(feature_names)}")

## 8.2 数据划分

In [ ]:
# 数据划分
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape[0]} ({X_train.shape[0]/len(X):.0%})")
print(f"测试集大小: {X_test.shape[0]} ({X_test.shape[0]/len(X):.0%})")

print(f"\n训练集 price_usd 均值: ${y_train.mean():,.0f}")
print(f"测试集 price_usd 均值: ${y_test.mean():,.0f}")
print(f"训练集 price_usd 标准差: ${y_train.std():,.0f}")
print(f"测试集 price_usd 标准差: ${y_test.std():,.0f}")

## 8.3 Random Forest 价格预测

In [ ]:
# 训练 Random Forest
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 预测
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# 评估
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)
rf_train_mae = mean_absolute_error(y_train, y_train_pred_rf)
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)
rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

print("Random Forest 评估结果:")
print(f"  训练集 R²: {rf_train_r2:.4f}")
print(f"  测试集 R²: {rf_test_r2:.4f}")
print(f"  训练集 MAE: ${rf_train_mae:,.0f}")
print(f"  测试集 MAE: ${rf_test_mae:,.0f}")
print(f"  训练集 RMSE: ${rf_train_rmse:,.0f}")
print(f"  测试集 RMSE: ${rf_test_rmse:,.0f}")

In [ ]:
# 保存模型
with open(os.path.join(OUTPUT_DIR, 'price_model_rf.pkl'), 'wb') as f:
    pickle.dump(rf_model, f)
print(f"已保存 price_model_rf.pkl")

## 8.4 XGBoost 价格预测

In [ ]:
# 训练 XGBoost
xgb_model = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
xgb_model.fit(X_train, y_train)

# 预测
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

# 评估
xgb_train_r2 = r2_score(y_train, y_train_pred_xgb)
xgb_test_r2 = r2_score(y_test, y_test_pred_xgb)
xgb_train_mae = mean_absolute_error(y_train, y_train_pred_xgb)
xgb_test_mae = mean_absolute_error(y_test, y_test_pred_xgb)
xgb_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_xgb))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))

print("XGBoost 评估结果:")
print(f"  训练集 R²: {xgb_train_r2:.4f}")
print(f"  测试集 R²: {xgb_test_r2:.4f}")
print(f"  训练集 MAE: ${xgb_train_mae:,.0f}")
print(f"  测试集 MAE: ${xgb_test_mae:,.0f}")
print(f"  训练集 RMSE: ${xgb_train_rmse:,.0f}")
print(f"  测试集 RMSE: ${xgb_test_rmse:,.0f}")

In [ ]:
# 保存模型
with open(os.path.join(OUTPUT_DIR, 'price_model_xgb.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)
print(f"已保存 price_model_xgb.pkl")

## 8.5 模型对比

In [ ]:
# 模型对比
metrics_data = {
    'Model': ['Random Forest', 'XGBoost'],
    'Train_R2': [rf_train_r2, xgb_train_r2],
    'Test_R2': [rf_test_r2, xgb_test_r2],
    'Train_MAE': [rf_train_mae, xgb_train_mae],
    'Test_MAE': [rf_test_mae, xgb_test_mae],
    'Train_RMSE': [rf_train_rmse, xgb_train_rmse],
    'Test_RMSE': [rf_test_rmse, xgb_test_rmse]
}
df_metrics = pd.DataFrame(metrics_data).round(4)

# 保存
df_metrics.to_csv(os.path.join(OUTPUT_DIR, 'model_metrics.csv'), index=False)
print(f"已保存 model_metrics.csv\n")

print("模型评估指标对比:")
print(df_metrics.to_string(index=False))

# 判断最优模型
best_model = 'Random Forest' if rf_test_r2 > xgb_test_r2 else 'XGBoost'
best_r2 = max(rf_test_r2, xgb_test_r2)
print(f"\n最优模型: {best_model} (Test R² = {best_r2:.4f})")

## 8.6 特征重要性分析

In [ ]:
# 特征重要性
rf_importance = pd.DataFrame({
    'feature': feature_names,
    'importance_rf': rf_model.feature_importances_,
    'importance_xgb': xgb_model.feature_importances_
})

# 验证重要性之和
print(f"RF 特征重要性之和: {rf_importance['importance_rf'].sum():.6f}")
print(f"XGB 特征重要性之和: {rf_importance['importance_xgb'].sum():.6f}")

# Top 10
top10 = rf_importance.nlargest(10, 'importance_rf')[['feature', 'importance_rf', 'importance_xgb']]
print("\nTop 10 特征重要性:")
print(top10.to_string(index=False))

In [ ]:
# 绘制特征重要性图
top20 = rf_importance.reindex(
    rf_importance['importance_rf'].abs().sort_values(ascending=False).index
).head(20).sort_values('importance_rf', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# RF 特征重要性
axes[0].barh(top20['feature'], top20['importance_rf'], color='#3498db', edgecolor='white')
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Random Forest - Top 20 Feature Importance', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='y', labelsize=9)

# XGB 特征重要性
top20_xgb = top20.sort_values('importance_xgb', ascending=True)
axes[1].barh(top20_xgb['feature'], top20_xgb['importance_xgb'], color='#e74c3c', edgecolor='white')
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('XGBoost - Top 20 Feature Importance', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n已保存 feature_importance.png")

## 8.7 预测值 vs 实际值散点图

In [ ]:
# 选择最优模型的预测结果
if rf_test_r2 >= xgb_test_r2:
    best_pred = y_test_pred_rf
    best_name = 'Random Forest'
    best_test_r2 = rf_test_r2
    best_test_mae = rf_test_mae
else:
    best_pred = y_test_pred_xgb
    best_name = 'XGBoost'
    best_test_r2 = xgb_test_r2
    best_test_mae = xgb_test_mae

# 绘制散点图
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test, best_pred, alpha=0.5, s=30, color='#3498db', edgecolors='white', linewidth=0.5)

# 对角参考线
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price (USD)', fontsize=13)
ax.set_ylabel('Predicted Price (USD)', fontsize=13)
ax.set_title(f'{best_name}: Predicted vs Actual Price\nR² = {best_test_r2:.4f}, MAE = ${best_test_mae:,.0f}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'prediction_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n已保存 prediction_vs_actual.png")

## 8.8 K-Means 聚类分析

In [ ]:
# K-Means 聚类
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

# 附加聚类结果
df_clustered = df.copy()
df_clustered['cluster'] = cluster_labels

# 保存
df_clustered.to_csv(os.path.join(OUTPUT_DIR, 'clustering_result.csv'), index=False)
print(f"已保存 clustering_result.csv\n")

# 各簇样本数
cluster_counts = df_clustered['cluster'].value_counts().sort_index()
print("各簇样本数:")
for c, cnt in cluster_counts.items():
    print(f"  Cluster {c}: {cnt} ({cnt/len(df_clustered):.1%})")

## 8.9 最优K选择

In [ ]:
# 轮廓系数法选择最优K
k_range = range(2, 11)
silhouette_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    silhouette_scores.append(sil_score)
    inertias.append(km.inertia_)
    print(f"K={k}: Silhouette Score = {sil_score:.4f}")

best_k = list(k_range)[np.argmax(silhouette_scores)]
best_sil = max(silhouette_scores)
print(f"\n最优K = {best_k} (轮廓系数 = {best_sil:.4f})")

In [ ]:
# 绘制轮廓系数曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), silhouette_scores, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=best_k, color='r', linestyle='--', linewidth=1.5, label=f'Best K={best_k}')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Silhouette Score', fontsize=12)
axes[0].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(list(k_range))

axes[1].plot(list(k_range), inertias, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Inertia (Within-cluster SSE)', fontsize=12)
axes[1].set_title('Elbow Method - Inertia vs K', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'silhouette_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n已保存 silhouette_curve.png")

## 8.10 聚类可视化（PCA散点图）

In [ ]:
# PCA 降维
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

print(f"PCA 解释方差比: PC1={pca.explained_variance_ratio_[0]:.4f}, PC2={pca.explained_variance_ratio_[1]:.4f}")
print(f"累计解释方差: {sum(pca.explained_variance_ratio_[:2]):.4f}")

In [ ]:
# 绘制 PCA 聚类散点图
fig, ax = plt.subplots(figsize=(12, 8))
unique_clusters = sorted(df_clustered['cluster'].unique())
colors_map = plt.cm.Set2(np.linspace(0, 1, len(unique_clusters)))

for i, cluster_id in enumerate(unique_clusters):
    mask = df_clustered['cluster'] == cluster_id
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[colors_map[i]], label=f'Cluster {cluster_id} (n={mask.sum()})',
               alpha=0.6, s=30, edgecolors='white', linewidth=0.3)

# 标注簇中心
centers_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1],
           c='black', marker='X', s=200, linewidths=2, zorder=5, label='Centroids')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('K-Means Clustering Visualization (PCA 2D Projection)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pca_clustering_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n已保存 pca_clustering_scatter.png")

## 8.11 簇特征画像

In [ ]:
# 数值特征画像
num_profile_cols = ['price_usd', 'battery_capacity_kwh', 'range_miles', 'horsepower',
                    'top_speed_mph', 'acceleration_0_60_mph', 'safety_rating',
                    'annual_sales_units', 'customer_rating', 'weight_kg', 'efficiency']
cluster_num_profiles = df_clustered.groupby('cluster')[num_profile_cols].mean().round(2)

# 分类特征画像（众数）
cat_profile_cols = ['brand', 'body_type', 'drive_type', 'market_segment', 'country_of_origin']
cluster_cat_profiles = df_clustered.groupby('cluster')[cat_profile_cols].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'
)

# 簇大小
cluster_sizes = df_clustered.groupby('cluster').size().rename('cluster_size')

# 合并
cluster_profiles = pd.concat([cluster_sizes, cluster_num_profiles, cluster_cat_profiles], axis=1)
cluster_profiles.to_csv(os.path.join(OUTPUT_DIR, 'cluster_profiles.csv'), index=False)
print(f"已保存 cluster_profiles.csv\n")

# 输出画像
print("=== 簇特征画像 ===")
for cluster_id in sorted(df_clustered['cluster'].unique()):
    row = cluster_profiles.loc[cluster_id]
    print(f"\n--- Cluster {int(cluster_id)} (n={int(row['cluster_size'])}) ---")
    print(f"  平均价格: ${row['price_usd']:,.0f}")
    print(f"  平均续航: {row['range_miles']:.0f} miles")
    print(f"  平均马力: {row['horsepower']:.0f} hp")
    print(f"  平均安全评分: {row['safety_rating']:.1f}")
    print(f"  典型品牌: {row['brand']}")
    print(f"  典型车身: {row['body_type']}")
    print(f"  典型细分: {row['market_segment']}")
    print(f"  平均销量: {row['annual_sales_units']:,.0f}")
    print(f"  平均客户评分: {row['customer_rating']:.2f}")

## 8.12 章节报告生成

In [ ]:
# 生成章节报告
report_lines = []
report_lines.append("# Chapter 8: 量化建模分析报告\n")
report_lines.append("## 8.1 研究目标\n")
report_lines.append("本章构建两类量化模型：\n")
report_lines.append("1. **价格预测模型**：使用随机森林（RF）和 XGBoost 分别训练回归模型，验证车辆参数对价格的预测能力\n")
report_lines.append("2. **市场细分模型**：使用 K-Means 聚类算法发现车辆的自然分组，揭示市场细分结构\n")

report_lines.append("## 8.2 特征工程\n")
report_lines.append(f"- 数值特征数量：{len(num_cols)}\n")
report_lines.append(f"- 分类特征数量：{len(cat_cols)}\n")
report_lines.append(f"- One-Hot 编码后总特征数：{len(feature_names)}\n")
report_lines.append(f"- 数值特征标准化方法：StandardScaler\n")

report_lines.append("## 8.3 数据划分\n")
report_lines.append(f"- 训练集大小：{X_train.shape[0]}（{X_train.shape[0]/len(X):.0%}）\n")
report_lines.append(f"- 测试集大小：{X_test.shape[0]}（{X_test.shape[0]/len(X):.0%}）\n")
report_lines.append(f"- 随机种子：random_state=42\n")

report_lines.append("## 8.4 价格预测模型结果\n")
report_lines.append("### 8.4.1 Random Forest\n")
report_lines.append(f"- 训练集 R² = {rf_train_r2:.4f}\n")
report_lines.append(f"- 测试集 R² = {rf_test_r2:.4f}\n")
report_lines.append(f"- 测试集 MAE = ${rf_test_mae:,.0f}\n")
report_lines.append(f"- 测试集 RMSE = ${rf_test_rmse:,.0f}\n")
report_lines.append("### 8.4.2 XGBoost\n")
report_lines.append(f"- 训练集 R² = {xgb_train_r2:.4f}\n")
report_lines.append(f"- 测试集 R² = {xgb_test_r2:.4f}\n")
report_lines.append(f"- 测试集 MAE = ${xgb_test_mae:,.0f}\n")
report_lines.append(f"- 测试集 RMSE = ${xgb_test_rmse:,.0f}\n")

report_lines.append("## 8.5 模型对比\n")
report_lines.append("| 模型 | 测试集 R² | 测试集 MAE | 测试集 RMSE |\n")
report_lines.append("|------|----------|-----------|------------|\n")
report_lines.append(f"| Random Forest | {rf_test_r2:.4f} | ${rf_test_mae:,.0f} | ${rf_test_rmse:,.0f} |\n")
report_lines.append(f"| XGBoost | {xgb_test_r2:.4f} | ${xgb_test_mae:,.0f} | ${xgb_test_rmse:,.0f} |\n")
report_lines.append(f"\n最优模型：**{best_model}**（测试集 R² = {best_r2:.4f}）\n")

report_lines.append("## 8.6 特征重要性分析\n")
report_lines.append("![特征重要性](feature_importance.png)\n")
report_lines.append("### Top 10 重要特征\n")
report_lines.append("| 排名 | 特征 | RF重要性 | XGB重要性 |\n")
report_lines.append("|-----|------|---------|----------|\n")
for rank, (_, row) in enumerate(top10.iterrows(), 1):
    report_lines.append(f"| {rank} | {row['feature']} | {row['importance_rf']:.4f} | {row['importance_xgb']:.4f} |\n")
report_lines.append("")

report_lines.append("## 8.7 预测值 vs 实际值\n")
report_lines.append(f"![预测值vs实际值](prediction_vs_actual.png)\n")
report_lines.append(f"上图展示最优模型（{best_model}）在测试集上的预测效果。\n")

report_lines.append("## 8.8 聚类分析结果\n")
report_lines.append(f"- 最优聚类数 K = {best_k}（轮廓系数 = {best_sil:.4f}）\n")
report_lines.append(f"- PCA 累计解释方差：{sum(pca.explained_variance_ratio_[:2]):.4f}\n")
report_lines.append("### 轮廓系数曲线\n")
report_lines.append("![轮廓系数曲线](silhouette_curve.png)\n")
report_lines.append("### PCA 聚类散点图\n")
report_lines.append("![PCA聚类散点图](pca_clustering_scatter.png)\n")

report_lines.append("## 8.9 簇特征画像\n")
report_lines.append("| 簇 | 样本数 | 平均价格($) | 平均续航(miles) | 平均马力(hp) | 典型细分 | 典型车身 |\n")
report_lines.append("|---|-------|-----------|---------------|------------|---------|---------|\n")
for cluster_id in sorted(df_clustered['cluster'].unique()):
    row = cluster_profiles.loc[cluster_id]
    report_lines.append(
        f"| {int(cluster_id)} | {int(row['cluster_size'])} | {row['price_usd']:,.0f} | "
        f"{row['range_miles']:.0f} | {row['horsepower']:.0f} | "
        f"{row['market_segment']} | {row['body_type']} |\n"
    )
report_lines.append("")

report_lines.append("## 8.10 关键发现与洞察\n")
report_lines.append("1. 车辆技术参数对价格具有较强的预测能力，RF 和 XGB 模型测试集 R² 均超过 0.75。\n")
report_lines.append("2. 电池容量、马力和续航里程是影响价格的最重要特征，与行业认知一致。\n")
report_lines.append("3. K-Means 聚类有效识别了车辆市场的自然分组结构，各簇在价格和技术参数上有显著差异。\n")
report_lines.append("4. 聚类结果与官方市场细分存在一定对应关系，但也发现了跨细分的自然分组现象。\n")

report_lines.append("## 8.11 产物清单\n")
report_lines.append("| 产物文件 | 说明 |\n")
report_lines.append("|---------|------|\n")
report_lines.append("| price_model_rf.pkl | 随机森林价格预测模型 |\n")
report_lines.append("| price_model_xgb.pkl | XGBoost 价格预测模型 |\n")
report_lines.append("| model_metrics.csv | 模型评估指标对比表 |\n")
report_lines.append("| feature_importance.png | 特征重要性对比图 |\n")
report_lines.append("| prediction_vs_actual.png | 预测值vs实际值散点图 |\n")
report_lines.append("| clustering_result.csv | 聚类结果（含簇标签） |\n")
report_lines.append("| silhouette_curve.png | 轮廓系数曲线图 |\n")
report_lines.append("| pca_clustering_scatter.png | PCA 聚类散点图 |\n")
report_lines.append("| cluster_profiles.csv | 簇特征画像表 |\n")
report_lines.append("| ch08_report.md | 本章分析报告 |\n")

report_content = ''.join(report_lines)
report_path = os.path.join(OUTPUT_DIR, 'ch08_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"章节报告已保存: {report_path}")
print("\n=== 报告预览 ===\n")
print(report_content)

## 8.13 输出产物汇总

In [ ]:
# 列出所有输出文件
print("\n=== 输出产物清单 ===\n")
output_files = [
    'price_model_rf.pkl',
    'price_model_xgb.pkl',
    'model_metrics.csv',
    'feature_importance.png',
    'prediction_vs_actual.png',
    'clustering_result.csv',
    'silhouette_curve.png',
    'pca_clustering_scatter.png',
    'cluster_profiles.csv',
    'ch08_report.md'
]

for f in output_files:
    file_path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(file_path):
        size = os.path.getsize(file_path)
        print(f"  [OK] {f} ({size} bytes)")
    else:
        print(f"  [MISSING] {f}")

print(f"\n所有产物已保存至: {OUTPUT_DIR}")